In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
import json
with open("../citation.json", 'r') as f:
    citation = json.load(f)
citation

{'doi': 'https://doi.org/10.1161/CIRCULATIONAHA.119.045401',
 'citation': 'Tucker, N.R., Chaffin, M., Fleming, S.J., Hall, A.W., Parsons, V.A., Bedi Jr, K.C., Akkad, A.D., Herndon, C.N., Arduini, A., Papangeli, I. and Roselli, C., 2020. Transcriptional and cellular diversity of the human heart. Circulation, 142(5), pp.466-482.',
 'tables': ['https://www.ahajournals.org/action/downloadSupplement?doi=10.1161%2FCIRCULATIONAHA.119.045401&file=supplemental+tables+%282%29.xlsx']}

## Define the metadata

In [3]:
organism = "homo_sapiens"
cell_source = "heart"
cell_state = None

reference = "GRCh38"

table_name = "clean_degs.xlsx" # local name

## Read in file

In [4]:
excel = pd.read_excel(table_name, sheet_name=None)

table = excel["Table IV MarkerGene"].rename(columns = {"Cell Type": "celltype_id", "Gene": "gene", })
df = table[:-1].copy()
df["celltype"] = df.celltype_id.apply(lambda x: x.split('. ')[-1] )

In [5]:
df.head()

,celltype_id,gene,Ensembl ID,Chromosome,Pct.Target,Pct.Other,avg_logFC,AUC,PPV50,Marker,celltype
0,1. Fibroblast I,DCN,ENSG00000011465,12,0.888,0.350,1.267279,0.853,0.717110,1.0,Fibroblast I
1,1. Fibroblast I,LAMA2,ENSG00000196569,6,0.993,0.808,0.851194,0.847,0.551188,1.0,Fibroblast I
2,1. Fibroblast I,NEGR1,ENSG00000172260,1,0.905,0.371,1.251984,0.842,0.709009,1.0,Fibroblast I
3,1. Fibroblast I,ACSM3,ENSG00000005187,16,0.836,0.311,1.473957,0.841,0.728646,1.0,Fibroblast I
4,1. Fibroblast I,ABCA6,ENSG00000154262,17,0.754,0.280,1.404422,0.804,0.729496,1.0,Fibroblast I


## Unfiltered

In [6]:
unfiltered_df = df.copy()
unfiltered_df.rename(columns ={"celltype": "cell_type_label"}, inplace=True)
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["gene_id"] = None

unfiltered_df.drop(['celltype_id', 'Ensembl ID', 'Chromosome', 'Pct.Target', 'Pct.Other', 'avg_logFC', 'AUC', 'PPV50', 'Marker'], axis=1, inplace=True) 
save = unfiltered_df.to_dict(orient = "records")

## Save unfiltered

In [7]:
with open("evidence_unfiltered.json", 'w') as f:
    json.dump(save, f, indent=4)

## Perform the filtering

In [6]:
col_pval = "p_val_adj"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "celltype"
col_rank = "Pct.Target"

In [7]:
min_mean = 10
max_pval = 0.05
min_lfc = 1
max_gene_shares = 10
max_per_celltype = 20

# filter by pval and lfc
# dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")
dfc = df.query(f"Marker == 1.0 & {col_lfc} >= {min_lfc}")


# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

if col_rank:
    m = dfc[dfc[col_gene].isin(non_repeat_genes)].sort_values(col_rank, ascending = True)
else:
    m = dfc[dfc[col_gene].isin(non_repeat_genes)]

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [8]:
markers[col_ct].value_counts()

celltype
9.Endothelium I                  20
Macrophage                       20
Vascular Smooth Muscle           20
Fibroblast III                   20
Endothelium II                   20
Ventricular Cardiomyocyte II     20
Ventricular Cardiomyocyte III    20
Adipocyte                        20
Atrial Cardiomyocyte             20
Ventricular Cardiomyocyte I      20
Pericyte                         18
Neuronal                         12
Fibroblast II                    12
Lymphocyte                        8
Fibroblast I                      8
Name: count, dtype: int64

In [9]:
if col_rank:
    print(markers.groupby(col_ct)[col_rank].mean().sort_values())

celltype
Lymphocyte                       0.564375
Neuronal                         0.617083
Pericyte                         0.656611
9.Endothelium I                  0.676100
Macrophage                       0.684300
Vascular Smooth Muscle           0.713900
Fibroblast III                   0.742450
Endothelium II                   0.753350
Fibroblast I                     0.807875
Fibroblast II                    0.821833
Ventricular Cardiomyocyte II     0.865550
Adipocyte                        0.872800
Ventricular Cardiomyocyte III    0.889950
Atrial Cardiomyocyte             0.932250
Ventricular Cardiomyocyte I      0.980800
Name: Pct.Target, dtype: float64


In [10]:
markers[col_ct].value_counts()

celltype
9.Endothelium I                  20
Macrophage                       20
Vascular Smooth Muscle           20
Fibroblast III                   20
Endothelium II                   20
Ventricular Cardiomyocyte II     20
Ventricular Cardiomyocyte III    20
Adipocyte                        20
Atrial Cardiomyocyte             20
Ventricular Cardiomyocyte I      20
Pericyte                         18
Neuronal                         12
Fibroblast II                    12
Lymphocyte                        8
Fibroblast I                      8
Name: count, dtype: int64

## Find Ensembl ID


In [11]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [12]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=5):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [13]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json

In [14]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "gene"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["gene_id"] = df["gene"]
df["gene_id"] = df["gene_id"].apply(run)
save = df.to_dict(orient = "records")

In [15]:
save

[{'cell_type_label': 'Lymphocyte',
  'gene': 'IKZF1',
  'organism': 'homo_sapiens',
  'cell_source': 'heart',
  'cell_state': None,
  'gene_id': 'ENSG00000185811'},
 {'cell_type_label': 'Neuronal',
  'gene': 'GPM6B',
  'organism': 'homo_sapiens',
  'cell_source': 'heart',
  'cell_state': None,
  'gene_id': 'ENSG00000046653'},
 {'cell_type_label': 'Neuronal',
  'gene': 'SHISA9',
  'organism': 'homo_sapiens',
  'cell_source': 'heart',
  'cell_state': None,
  'gene_id': 'ENSG00000237515'},
 {'cell_type_label': 'Neuronal',
  'gene': 'SLC35F1',
  'organism': 'homo_sapiens',
  'cell_source': 'heart',
  'cell_state': None,
  'gene_id': 'ENSG00000196376'},
 {'cell_type_label': '9.Endothelium I',
  'gene': 'PECAM1',
  'organism': 'homo_sapiens',
  'cell_source': 'heart',
  'cell_state': None,
  'gene_id': 'ENSG00000261371'},
 {'cell_type_label': 'Pericyte',
  'gene': 'CARMN',
  'organism': 'homo_sapiens',
  'cell_source': 'heart',
  'cell_state': None,
  'gene_id': 'ENSG00000249669'},
 {'cell_t

## Save evidence

In [16]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 